# LangGraph 기초 + MLflow 트레이싱
상태(State) · 노드(Node) · 엣지(Edge) 로 워크플로를 만들고, 마지막에 `create_agent` 로 도구 에이전트를 만든다.

# **1. 환경 설정**

In [1]:
from src.common import setup_mlflow, get_model

setup_mlflow(experiment = 'langgraph-study')
model = get_model()
model.model

'gemini-3.1-flash-lite'

# **2. StateGraph 기초 (LLM 없이)**
노드는 `state -> 부분 업데이트(dict)` 함수. 리듀서(`operator.add`)를 붙인 필드는 덮어쓰지 않고 누적된다.  
API 키 없이도 돌아가므로 그래프 구조부터 익히기 좋다.

In [2]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

class CountState(TypedDict):
    n: int
    log: Annotated[list[str], operator.add]   # 누적

def inc(state: CountState):
    return {"n": state["n"] + 1, "log": [f"inc -> {state['n'] + 1}"]}

def should_continue(state: CountState):
    return "inc" if state["n"] < 3 else END

builder = StateGraph(CountState)
builder.add_node("inc", inc)
builder.add_edge(START, "inc")
builder.add_conditional_edges("inc", should_continue, ["inc", END])
counter = builder.compile()

counter.invoke({"n": 0, "log": []})

{'n': 3, 'log': ['inc -> 1', 'inc -> 2', 'inc -> 3']}

In [3]:
# 그래프 구조 확인 (Mermaid)
print(counter.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	inc(inc)
	__end__([<p>__end__</p>]):::last
	__start__ --> inc;
	inc -.-> __end__;
	inc -.-> inc;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



# **3. LLM 노드 + 조건 분기 (라우팅)**
질문을 분류한 뒤 분류 결과에 따라 다른 노드로 보낸다.

In [4]:
from typing import Literal
from pydantic import BaseModel

class Route(BaseModel):
    kind: Literal["code", "concept"]

class QAState(TypedDict):
    question: str
    kind: str
    answer: str

router = model.with_structured_output(Route)

def classify(state: QAState):
    r = router.invoke(f"다음 질문이 코드 작성 요청이면 code, 개념 설명이면 concept: {state['question']}")
    return {"kind": r.kind}

def answer_code(state: QAState):
    return {"answer": model.invoke(f"파이썬 코드로만 답해: {state['question']}").text}

def answer_concept(state: QAState):
    return {"answer": model.invoke(f"비유를 들어 세 줄로 설명해: {state['question']}").text}

b = StateGraph(QAState)
b.add_node("classify", classify)
b.add_node("code", answer_code)
b.add_node("concept", answer_concept)
b.add_edge(START, "classify")
b.add_conditional_edges("classify", lambda s: s["kind"], ["code", "concept"])
b.add_edge("code", END)
b.add_edge("concept", END)
qa = b.compile()

out = qa.invoke({"question": "리스트를 역순으로 정렬하는 법"})
print(out["kind"]); print(out["answer"])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


code
```python
# 방법 1: 리스트 자체를 수정 (in-place)
my_list = [1, 2, 3, 4, 5]
my_list.reverse()

# 방법 2: 새로운 리스트 생성
my_list = [1, 2, 3, 4, 5]
reversed_list = my_list[::-1]

# 방법 3: sorted() 함수 사용
my_list = [1, 2, 3, 4, 5]
reversed_list = sorted(my_list, reverse=True)
```


# **4. create_agent: 도구 에이전트 + 메모리**
`create_agent` 는 LangGraph 위에서 "모델 → 도구 실행 → 모델 …" 루프를 만들어 준다.  
`checkpointer` 를 주면 `thread_id` 별로 대화가 이어진다.

In [5]:
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def multiply(a: float, b: float) -> float:
    """두 수를 곱한다."""
    return a * b

@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 알려준다."""
    return f"{city}: 맑음, 21도"  # 실습용 가짜 데이터

agent = create_agent(
    model,
    tools=[multiply, get_weather],
    system_prompt="너는 한국어로 답하는 조수야. 필요하면 도구를 써.",
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "study-1"}}

r = agent.invoke({"messages": [{"role": "user", "content": "부산 날씨 어때? 그리고 12.5 곱하기 8 은?"}]}, config)
print(r["messages"][-1].text)

현재 부산은 맑고 기온은 21도입니다. 그리고 12.5 곱하기 8은 100입니다.


In [6]:
# 같은 thread_id 로 이어서 질문 → 이전 대화를 기억
r = agent.invoke({"messages": [{"role": "user", "content": "방금 계산한 값에 2를 곱하면?"}]}, config)
print(r["messages"][-1].text)

방금 계산한 100에 2를 곱하면 200입니다.


# **5. MLflow 에서 트레이스 확인**
에이전트 트레이스를 열면 모델 호출 → 도구 실행 → 모델 호출이 트리로 보인다.

In [7]:
import mlflow

mlflow.search_traces(max_results=10)[["trace_id", "request_time", "state", "execution_duration"]]

,trace_id,request_time,state,execution_duration
0,tr-379e7212941181fba31839a74e717fbf,1790642739535,IN_PROGRESS,1451
1,tr-b5ef6205cf4556e9c82492f65d538984,1790642738070,OK,1450
2,tr-795e8bbc9eec26840b726a4d99c87387,1790642735776,OK,2183
3,tr-0bb46c6617070884e2ef81432f87aeb6,1790642735758,OK,6
4,tr-4702d8b22b22b80cb61f0ec2a04aa671,1790585019414,OK,2005
5,tr-f30c0f17b1623ea37af41928321198c3,1790585017665,OK,1739
6,tr-cd134317549df20aaac7f260e4893ce0,1790585015296,OK,2308
7,tr-344057714d83e87cc850fa99fc483356,1790585015282,OK,2


# **99. 참고자료**
- LangGraph 문서: https://docs.langchain.com/oss/python/langgraph/overview
- create_agent: https://docs.langchain.com/oss/python/langchain/agents
- MLflow LangGraph 트레이싱: https://mlflow.org/docs/latest/genai/tracing/integrations/listing/langgraph